# Lesson 03: Edge AI Development Stack — Colab T4 GPU, Open-Source Runtimes & Google AI Studio

In this companion notebook, you will:
1. Inspect your Google Colab runtime hardware (CPU cores, RAM, and NVIDIA T4 GPU VRAM).
2. Configure the **Google AI Studio Free Tier** (`GEMINI_API_KEY` via `google-genai`) with an automatic offline fallback.
3. Install open-source Edge AI runtimes (`transformers`, `onnxruntime`, `huggingface_hub`) and run a memory-budget profiler to match SLMs to your device.

In [ ]:
!pip install -q -U google-genai transformers onnxruntime huggingface_hub psutil

## 1. Inspect Hardware & Available Memory
Before pulling any Small Language Model, verify whether Colab assigned a CPU or T4 GPU runtime and check available RAM/VRAM.

In [ ]:
import os
import platform
import psutil

ram_gb = psutil.virtual_memory().total / (1024 ** 3)
print(f"Platform: {platform.system()} {platform.machine()}")
print(f"CPU Cores: {os.cpu_count()} | System RAM: {ram_gb:.2f} GB")

try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        print(f"GPU Detected: {gpu_name} ({vram_gb:.2f} GB VRAM)")
    else:
        print("No GPU detected — running on CPU tier.")
except ImportError:
    print("PyTorch not installed; skipping CUDA check.")

## 2. Configure Google AI Studio Free Tier (`GEMINI_API_KEY`)
Add your free API key from [aistudio.google.com/apikey](https://aistudio.google.com/apikey) to Colab **Secrets** (`GEMINI_API_KEY`). If no key is set, the helper below falls back cleanly to a deterministic local response.

In [ ]:
GEMINI_API_KEY = None
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

if GEMINI_API_KEY:
    from google import genai
    client = genai.Client(api_key=GEMINI_API_KEY)
    resp = client.models.generate_content(
        model="gemini-2.5-flash",
        contents="In one sentence, explain why INT4 quantization helps Small Language Models run on edge devices."
    )
    print("Gemini Teacher Response:", resp.text.strip())
else:
    print("GEMINI_API_KEY not found — offline fallback active:")
    print("INT4 quantization reduces model weight memory by ~75%, letting multi-billion-parameter SLMs fit inside consumer RAM and NPU caches.")

## 3. Edge Memory Budget Calculator & Open-Source Tokenizer Test
Compute exact weight + KV-cache memory footprints for `Qwen2.5-0.5B-Instruct`, `Gemma-3n-E2B`, and `Phi-4-mini-3.8B`, then verify ONNX Runtime providers.

In [ ]:
import onnxruntime as ort

def estimate_memory_gb(params_b: float, weight_bits: int, layers: int, kv_heads: int, head_dim: int, seq_len: int, kv_bits: int = 16) -> float:
    weights_bytes = params_b * 1e9 * (weight_bits / 8.0)
    kv_bytes = 2 * layers * kv_heads * head_dim * seq_len * (kv_bits / 8.0)
    return (weights_bytes + kv_bytes) / (1024 ** 3)

candidates = [
    ("Qwen2.5-0.5B (INT4)", 0.5, 4, 24, 2, 64, 4096),
    ("Gemma-3n-E2B (INT4)", 2.0, 4, 30, 4, 256, 4096),
    ("Phi-4-mini-3.8B (INT4)", 3.8, 4, 32, 8, 128, 4096),
    ("Phi-4-mini-3.8B (FP16)", 3.8, 16, 32, 8, 128, 4096),
]

for name, p, wb, l, kvh, hd, s in candidates:
    mem = estimate_memory_gb(p, wb, l, kvh, hd, s)
    print(f"{name:26s} -> {mem:.2f} GB")

print("Available ONNX Runtime Execution Providers:", ort.get_available_providers())